---
title: "ESCUELA POLITECNICA NACIONAL"
subtitle: "PRACTICA #2: Practica Muestra vs Poblacion"
author: "Aidan Carrasco"
date: today
lang: es
bibliography: ref.bib
format: 
  pdf:
    papersize: letter
    margin-left: 2.5cm
    margin-right: 2.5cm
    margin-top: 3cm
    margin-bottom: 3cm
header-includes:
- \usepackage{fancyhdr}
- \pagestyle{fancy}
- \fancyhead[L]{Ciencia de Datos e Inteligencia Artificial}
- \fancyhead[R]{EPN}
- \fancyfoot[C]{\thepage} 
---

| | |
|---|---|
| **Estudiante** | Aidan Carrasco — usuario de GitHub: [0-AHCA-0] |
| **Carrera** | Ciencia de Datos e Inteligencia Artificial |
| **Docente** | CHRISTIAN GONZALO SUAREZ COBOS |
| **Período académico** | 2026-B |
| **Fecha de entrega** | 19/10/2026 |
| **Repositorio (fork)** | https://github.com/0-AHCA-0/muestreo-poblacional |

# 1. Objetivos
## Objetivo General: 
* Clasificar las variables de la base de fidelización de RetailAndina S.A. justificando cada decisión con criterio estadístico y comparar tres diseños de muestreo frente al parámetro poblacional real.

## Objetivos Específicos:

* Clasificar cada variable de la base de clientes como numérica (continua o discreta) o categórica (nominal, binaria u ordinal) aplicando criterios de continuidad y jerarquía.

* Verificar en código (`dtypes` en Python y `str` en R, junto con el conteo de valores únicos) los tipos inferidos por cada lenguaje, declarando explícitamente el orden en las variables ordinales (`CategoricalDtype` y `factor(..., ordered = TRUE)`).

* Analizar el efecto del sesgo de selección cuando el archivo está ordenado por ciudad y evaluar la variabilidad natural del sorteo aleatorio a través de múltiples semillas reproducibles.

# 2. Introducción 
La gerencia de **RetailAndina S.A.**, cadena ecuatoriana de tiendas de consumo masivo, planteó dos necesidades concretas a su equipo de Business Intelligence. Por un lado, el programa de fidelización ha crecido y se requiere conocer a fondo qué tipo de variables almacena la base de clientes para saber qué operaciones son válidas: qué campos se pueden promediar, cuáles tienen un orden jerárquico y cuáles únicamente admiten conteos por categoría. Por otro lado, la gerencia busca estimar el ingreso mensual promedio de su clientela tomando muestras de $n = 20$ registros a partir de la base completa de $N = 400$ clientes, pero necesita evidencia clara de qué hace que una muestra sea realmente representativa.

Como analista de BI, este trabajo aborda ambos frentes combinando criterio estadístico y verificación práctica en **Python** y **R**. Realizar este diagnóstico previo importa por dos razones fundamentales para el negocio:

1. **Los datos mal clasificados llevan a análisis inválidos:** confiar ciegamente en el tipo de dato que infiere el software (`int64` u `object`) provoca errores graves. Por ejemplo, promediar un código identificador carece de sentido matemático, tratar una escala ordinal de satisfacción (1 a 5) como si fuera una magnitud continua asume falsamente que un nivel 4 equivale al doble de satisfacción que un nivel 2, y dejar una variable ordinal en texto plano hace que el programa ordene las categorías alfabéticamente (`Posgrado` antes que `Primaria`), distorsionando reportes y gráficos.
2. **Una muestra mal diseñada no se corrige con más filas:** la representatividad no depende solo del tamaño muestral ($n$), sino del mecanismo de selección. Si se toman registros por conveniencia al inicio de un archivo que viene ordenado por ciudad, toda la muestra queda concentrada en un único grupo (Quito). Incluso si se duplicara o quintuplicara el número de filas bajo ese mismo corte, el sesgo de selección seguiría intacto y la conclusión comercial continuaría equivocada.

# 3. Problemas Planteados


## Muestreo de Datos

### muestreo.py

#### Explicación
1. **Población y parámetro real:** Se carga la base completa `../data/clientes_400.csv` ($N = 400$) con `pandas` y se obtiene el ingreso mensual promedio real (`1330.72 USD`), además de la distribución real por ciudad (`200` en Quito, `120` en Guayaquil y `80` en Cuenca).
2. **Muestreo por conveniencia (`head`):** Se seleccionan las primeras 20 filas del archivo. Al estar ordenado por ciudad, este corte captura únicamente clientes de Quito (`20/20`).
3. **Muestreo aleatorio simple (`sample`):** Se sortean 20 clientes con igual probabilidad fijando `random_state=1`, y luego se repite el experimento con las semillas `1` a `5` para observar cuánto fluctúa la media muestral entre sorteos.
4. **Muestreo estratificado (`groupby` + `frac=0.05`):** Se agrupa por `ciudad` y se extrae el 5 % dentro de cada estrato, garantizando la proporción exacta de la población (`10` de Quito, `6` de Guayaquil y `4` de Cuenca).

In [ ]:
import pandas as pd

pd.set_option("display.width", 100)


def reporte(nombre, muestra, parametro):
    # Muestra el tamano, la media estimada y el conteo por ciudad para detectar sesgos
    n = len(muestra)
    media = muestra["ingresos_mensuales"].mean()
    comp = muestra["ciudad"].value_counts().to_dict()
    print(f"[{nombre}] n={n}  media={media:.2f} USD  "
          f"(parametro {parametro:.2f}, diferencia {media - parametro:+.2f})")
    print(f"    composicion por ciudad: {comp}")


# 1. Poblacion completa y parametro real (N = 400)
poblacion = pd.read_csv("../data/clientes_400.csv")
parametro = poblacion["ingresos_mensuales"].mean()
print(f"Poblacion: {poblacion.shape[0]} clientes x {poblacion.shape[1]} columnas")
print(f"Parametro poblacional - ingreso medio: {parametro:.2f} USD")
print("Ciudades en la poblacion:", poblacion["ciudad"].value_counts().to_dict())
print()

# 2. Muestra por conveniencia: primeras 20 filas del archivo
# Como el CSV esta ordenado por ciudad, este corte selecciona solo clientes de Quito
muestra_conv = poblacion.head(20)

# 3. Muestra aleatoria simple: sorteo equiprobable de 20 clientes (5 % de la base)
muestra_aas = poblacion.sample(20, random_state=1)

# Repeticion del sorteo con semillas 1 a 5 para observar la variabilidad muestral
print("Variabilidad del muestreo aleatorio simple (semillas 1 a 5):")
for semilla in range(1, 6):
    sorteo = poblacion.sample(20, random_state=semilla)
    media_sorteo = sorteo["ingresos_mensuales"].mean()
    print(f"  Semilla {semilla}: media = {media_sorteo:.2f} USD "
          f"(diferencia {media_sorteo - parametro:+.2f})")
print()

# 4. Muestra estratificada: 5 % dentro de cada ciudad manteniendo la proporcion poblacional
# Asegura exactamente 10 de Quito, 6 de Guayaquil y 4 de Cuenca
muestra_est = poblacion.groupby("ciudad", group_keys=False).sample(frac=0.05, random_state=1)

# 5. Comparacion final de los tres disenos frente al parametro
reporte("Conveniencia", muestra_conv, parametro)
reporte("Aleatorio simple", muestra_aas, parametro)
reporte("Estratificado", muestra_est, parametro)
print()

resumen = pd.DataFrame([
    {"diseno": "Conveniencia (head)", "n": len(muestra_conv),
     "media estimada": round(muestra_conv["ingresos_mensuales"].mean(), 2),
     "diferencia": round(muestra_conv["ingresos_mensuales"].mean() - parametro, 2)},
    {"diseno": "Aleatorio simple (sample)", "n": len(muestra_aas),
     "media estimada": round(muestra_aas["ingresos_mensuales"].mean(), 2),
     "diferencia": round(muestra_aas["ingresos_mensuales"].mean() - parametro, 2)},
    {"diseno": "Estratificado (groupby + frac)", "n": len(muestra_est),
     "media estimada": round(muestra_est["ingresos_mensuales"].mean(), 2),
     "diferencia": round(muestra_est["ingresos_mensuales"].mean() - parametro, 2)},
])
print(f"Parametro poblacional: {parametro:.2f} USD")
print(resumen.to_string(index=False))

Poblacion: 400 clientes x 8 columnas
Parametro poblacional - ingreso medio: 1330.72 USD
Ciudades en la poblacion: {'Quito': 200, 'Guayaquil': 120, 'Cuenca': 80}

Variabilidad del muestreo aleatorio simple (semillas 1 a 5):
  Semilla 1: media = 1336.00 USD (diferencia +5.28)
  Semilla 2: media = 1405.00 USD (diferencia +74.28)
  Semilla 3: media = 1235.00 USD (diferencia -95.72)
  Semilla 4: media = 1491.50 USD (diferencia +160.78)
  Semilla 5: media = 1288.00 USD (diferencia -42.72)

[Conveniencia] n=20  media=1348.50 USD  (parametro 1330.72, diferencia +17.78)
    composicion por ciudad: {'Quito': 20}
[Aleatorio simple] n=20  media=1336.00 USD  (parametro 1330.72, diferencia +5.28)
    composicion por ciudad: {'Cuenca': 8, 'Quito': 8, 'Guayaquil': 4}
[Estratificado] n=20  media=1287.50 USD  (parametro 1330.72, diferencia -43.22)
    composicion por ciudad: {'Quito': 10, 'Guayaquil': 6, 'Cuenca': 4}

Parametro poblacional: 1330.72 USD
                        diseno  n  media estimada  

### muestreo.R

#### Explicación
1. **Carga y parámetro poblacional:** Se importa `../data/clientes_400.csv` con `read.csv()` y se calcula la media poblacional de `ingresos_mensuales` con `mean()`, verificando con `table()` la cantidad de clientes por ciudad.
2. **Diseño por conveniencia y aleatorio simple:** Con `head(poblacion, 20)` se extrae el corte inicial sesgado hacia Quito; luego con `sample(nrow(poblacion), 20)` y `set.seed(1:5)` se evalúa el sorteo aleatorio simple y su variación entre semillas.
3. **Diseño estratificado y tabla resumen:** Se aplica un muestreo del 5 % dentro de cada ciudad conservando las proporciones reales (`10` Quito, `6` Guayaquil, `4` Cuenca) y se consolidan las tres estimaciones en un `data.frame` comparativo.

In [1]:
poblacion <- read.csv("../data/clientes_400.csv")

reporte <- function(nombre, muestra, parametro) {
  # Imprime tamano, media estimada y conteo por ciudad para detectar sesgo de seleccion
  media <- mean(muestra$ingresos_mensuales)
  cat(sprintf("[%s] n=%d  media=%.2f USD  (parametro %.2f, diferencia %+.2f)\n",
              nombre, nrow(muestra), media, parametro, media - parametro))
  cat("    composicion por ciudad: ")
  print(table(muestra$ciudad))
}

# 1. Poblacion completa y parametro real (N = 400)
cat(sprintf("Poblacion: %d clientes x %d columnas\n", nrow(poblacion), ncol(poblacion)))
parametro <- mean(poblacion$ingresos_mensuales)
cat(sprintf("Parametro poblacional - ingreso medio: %.2f USD\n", parametro))
cat("Ciudades en la poblacion: ")
print(table(poblacion$ciudad))
cat("\n")

# 2. Muestra por conveniencia: primeras 20 filas del archivo
# El archivo viene ordenado por ciudad, por lo que solo entran clientes de Quito
muestra_conv <- head(poblacion, 20)

# 3. Muestra aleatoria simple: sorteo equiprobable de 20 clientes (5 % del total)
set.seed(1)
muestra_aas <- poblacion[sample(nrow(poblacion), 20), ]

# Repeticion del sorteo con semillas 1 a 5 para ver como varia la estimacion
cat("Variabilidad del muestreo aleatorio simple (semillas 1 a 5):\n")
for (semilla in 1:5) {
  set.seed(semilla)
  sorteo <- poblacion[sample(nrow(poblacion), 20), ]
  media_sorteo <- mean(sorteo$ingresos_mensuales)
  cat(sprintf("  Semilla %d: media = %.2f USD (diferencia %+.2f)\n",
              semilla, media_sorteo, media_sorteo - parametro))
}
cat("\n")

# 4. Muestra estratificada: 5 % dentro de cada ciudad (afijacion proporcional)
# Garantiza 10 clientes de Quito, 6 de Guayaquil y 4 de Cuenca
set.seed(1)
if (requireNamespace("dplyr", quietly = TRUE)) {
  muestra_est <- dplyr::slice_sample(dplyr::group_by(poblacion, ciudad), prop = 0.05)
} else {
  grupos <- split(poblacion, poblacion$ciudad)
  muestra_est <- do.call(rbind, lapply(grupos, function(df) {
    df[sample(nrow(df), size = round(nrow(df) * 0.05)), ]
  }))
}

# 5. Comparacion final de los tres disenos
reporte("Conveniencia", muestra_conv, parametro)
reporte("Aleatorio simple", muestra_aas, parametro)
reporte("Estratificado", muestra_est, parametro)
cat("\n")

resumen <- data.frame(
  diseno = c("Conveniencia (head)", "Aleatorio simple (sample)", "Estratificado (slice_sample)"),
  n = c(nrow(muestra_conv), nrow(muestra_aas), nrow(muestra_est)),
  media_estimada = round(c(mean(muestra_conv$ingresos_mensuales),
                           mean(muestra_aas$ingresos_mensuales),
                           mean(muestra_est$ingresos_mensuales)), 2),
  diferencia = round(c(mean(muestra_conv$ingresos_mensuales) - parametro,
                       mean(muestra_aas$ingresos_mensuales) - parametro,
                       mean(muestra_est$ingresos_mensuales) - parametro), 2)
)
cat(sprintf("Parametro poblacional: %.2f USD\n", parametro))
print(resumen, row.names = FALSE)

Poblacion: 400 clientes x 8 columnas
Parametro poblacional - ingreso medio: 1330.72 USD
Ciudades en la poblacion: 
   Cuenca Guayaquil     Quito 
       80       120       200 

Variabilidad del muestreo aleatorio simple (semillas 1 a 5):
  Semilla 1: media = 1318.00 USD (diferencia -12.72)
  Semilla 2: media = 1287.50 USD (diferencia -43.22)
  Semilla 3: media = 1233.00 USD (diferencia -97.72)
  Semilla 4: media = 1385.50 USD (diferencia +54.78)
  Semilla 5: media = 1283.00 USD (diferencia -47.72)

[Conveniencia] n=20  media=1348.50 USD  (parametro 1330.72, diferencia +17.78)
    composicion por ciudad: 
Quito 
   20 
[Aleatorio simple] n=20  media=1318.00 USD  (parametro 1330.72, diferencia -12.72)
    composicion por ciudad: 
   Cuenca Guayaquil     Quito 
        5         7         8 
[Estratificado] n=20  media=1226.00 USD  (parametro 1330.72, diferencia -104.72)
    composicion por ciudad: 
   Cuenca Guayaquil     Quito 
        4         6        10 

Parametro poblacional: 133

## Clasificación de Variables  

### clasificacion_variables.py

### Explicación

#### 1. Estructura y tipos inferidos (`dtypes`)
Primero se carga `../data/clientes.csv` (`12` filas y `8` columnas) con `pandas` y se inspeccionan los tipos que el lenguaje asigna por defecto: `edad`, `numero_hijos`, `ingresos_mensuales` y `satisfaccion` ingresan como enteros (`int64`), mientras que `cliente_id`, `genero`, `nivel_educacion` y `ciudad` ingresan como texto (`object`).

#### 2. Verificación de valores únicos
Se recorren todas las columnas con `.unique()` y `len()`, comprobando que `genero` tiene **3 categorías** en la base (`Femenino`, `Masculino`, `Otro`), por lo que se clasifica como categórica nominal y no binaria.

#### 3. Declaración del orden en variables ordinales y prueba de escritura
Con `pd.CategoricalDtype(..., ordered=True)` se establece la jerarquía lógica de `nivel_educacion` (`Primaria < Secundaria < Universidad < Posgrado`) y de `satisfaccion` (`1 < 2 < 3 < 4 < 5`), verificando el valor mínimo con `.min()`. Además, se demuestra qué ocurre al escribir `"Postgrado"` con `t`: los 2 registros que no coinciden con la lista se convierten silenciosamente en `NaN`.

#### 4. Resumen de clasificación
Finalmente, se presenta el diccionario con la clasificación estadística definitiva de las 8 columnas de la tabla.

In [ ]:
import pandas as pd

clientes = pd.read_csv("../data/clientes.csv")

# 1. Estructura general del archivo
print(f"Dimensiones: {clientes.shape[0]} filas x {clientes.shape[1]} columnas\n")
print("Primeras filas:")
print(clientes.to_string(index=False))
print()

# 2. Tipos de datos que infiere pandas por defecto
# Llegan como entero (int64): edad, numero_hijos, ingresos_mensuales y satisfaccion.
# Llegan como texto (object): cliente_id, genero, nivel_educacion y ciudad.
# Aunque satisfaccion aparece como int64, en realidad representa una escala ordinal.
print("Tipos inferidos (dtype):")
print(clientes.dtypes)
print()

# 3. Revision de valores unicos por columna
# Permite ver que genero tiene 3 categorias en la base (Femenino, Masculino, Otro)
print("Valores unicos por variable:")
for columna in clientes.columns:
    unicos = clientes[columna].unique()
    print(f"  {columna} ({len(unicos)} unicos): {unicos.tolist()}")
print()

# 4. Definir el orden correcto en las variables ordinales
orden_educacion = pd.CategoricalDtype(
    categories=["Primaria", "Secundaria", "Universidad", "Posgrado"],
    ordered=True
)
clientes["nivel_educacion"] = clientes["nivel_educacion"].astype(orden_educacion)
print("Nivel educativo minimo:", clientes["nivel_educacion"].min())

orden_satisfaccion = pd.CategoricalDtype(
    categories=[1, 2, 3, 4, 5],
    ordered=True
)
clientes["satisfaccion"] = clientes["satisfaccion"].astype(orden_satisfaccion)
print("Satisfaccion minima registrada:", clientes["satisfaccion"].min())

# Prueba de error de escritura al declarar categorias ("Postgrado" con t)
# Los registros que no coinciden con la lista se vuelven NaN sin lanzar error
educacion_mal_escrita = pd.CategoricalDtype(
    categories=["Primaria", "Secundaria", "Universidad", "Postgrado"],
    ordered=True
)
faltantes_typo = pd.read_csv("../data/clientes.csv")["nivel_educacion"].astype(educacion_mal_escrita).isna().sum()
print(f"Valores perdidos (NaN) al escribir 'Postgrado' con t: {faltantes_typo}")
print()

# 5. Resumen final de la clasificacion de variables
clasificacion = {
    "cliente_id": "Categorica nominal (identificador)",
    "edad": "Numerica continua (registrada en anos cumplidos, discreta en el archivo)",
    "numero_hijos": "Numerica discreta",
    "genero": "Categorica nominal (3 categorias)",
    "nivel_educacion": "Categorica ordinal",
    "ciudad": "Categorica nominal",
    "ingresos_mensuales": "Numerica continua",
    "satisfaccion": "Categorica ordinal",
}

print("Resumen de la clasificacion:")
for variable, tipo in clasificacion.items():
    print(f"  {variable:20s} {tipo}")

| Variable | Tipo estadístico | Criterio de negocio | Verificación en Python (`dtype` y únicos) |
|---|---|---|---|
| `cliente_id` | Categórica nominal (identificador) | Etiqueta única por cliente; no tiene sentido promediarla. | `object`, 12 únicos (`C001`–`C012`) |
| `edad` | Numérica continua (discretizada) | El tiempo es continuo (admite fracciones de año), aunque se registra en años cumplidos enteros. | `int64`, 12 únicos (`22`–`48`) |
| `numero_hijos` | Numérica discreta | Conteo entero de personas; no existe la mitad de un hijo. | `int64`, 5 únicos (`0, 1, 2, 3, 4`) |
| `genero` | Categórica nominal | Categorías sin orden jerárquico; presenta 3 niveles en la base. | `object`, 3 únicos (`Femenino`, `Masculino`, `Otro`) |
| `nivel_educacion` | Categórica ordinal | Jerarquía educativa clara sin distancia numérica igual entre niveles. | `category` ordenada, `.min()` = `Primaria` |
| `ciudad` | Categórica nominal | Ubicación geográfica sin orden intrínseco. | `object`, 3 únicos (`Quito`, `Guayaquil`, `Cuenca`) |
| `ingresos_mensuales` | Numérica continua | Magnitud monetaria en USD donde la mitad del valor tiene pleno sentido. | `int64`, 12 únicos (`760`–`2450` USD) |
| `satisfaccion` | Categórica ordinal | Escala de opinión 1–5 con orden, donde 4 no representa el doble de 2. | Llega como `int64`, convertida a `category` ordenada (`1..5`), `.min()` = `2` |

### clasificacion_variables.R

## Explicación 

### 1. Lectura e inspección con `str()`
Se importa el archivo `../data/clientes.csv` usando `read.csv(..., stringsAsFactors = FALSE)` y se revisa su estructura con `str()`, identificando qué variables entran como `int` y cuáles como `chr`.

### 2. Conteo de categorías con `unique()` y `length()`
Mediante un bucle `for` sobre `names(clientes)` se listan los valores únicos de cada columna, verificando nuevamente los 3 niveles de `genero` y los 4 niveles presentes en `satisfaccion` y `nivel_educacion`.

### 3. Factores ordenados (`factor(..., ordered = TRUE)`)
Se declara el orden natural de `nivel_educacion` y `satisfaccion` usando factores ordenados en R, comprobando con `min()` que el nivel inferior es `Primaria` y `2` respectivamente, y verificando con `sum(is.na(...))` los `2` valores perdidos (`NA`) cuando se introduce el error tipográfico `"Postgrado"`.

In [ ]:
clientes <- read.csv("../data/clientes.csv", stringsAsFactors = FALSE)

# 1. Estructura general del archivo
cat(sprintf("Dimensiones: %d filas x %d columnas\n\n", nrow(clientes), ncol(clientes)))
print(clientes)
cat("\n")

# 2. Tipos de datos inferidos por R (str)
# Llegan como entero (int): edad, numero_hijos, ingresos_mensuales y satisfaccion.
# Llegan como texto (chr): cliente_id, genero, nivel_educacion y ciudad.
# La columna satisfaccion viene como int pero su naturaleza es categorica ordinal.
str(clientes)
cat("\n")

# 3. Revision de valores unicos por columna
# Confirmamos que genero tiene 3 categorias (Femenino, Masculino, Otro)
cat("Valores unicos por variable:\n")
for (columna in names(clientes)) {
  unicos <- unique(clientes[[columna]])
  cat(sprintf("  %s (%d unicos): %s\n", columna, length(unicos), paste(unicos, collapse = ", ")))
}
cat("\n")

# 4. Definir el orden de las variables ordinales con factores ordenados
educacion_original <- clientes$nivel_educacion

clientes$nivel_educacion <- factor(
  clientes$nivel_educacion,
  levels = c("Primaria", "Secundaria", "Universidad", "Posgrado"),
  ordered = TRUE
)
cat("Nivel educativo minimo:", as.character(min(clientes$nivel_educacion)), "\n")

clientes$satisfaccion <- factor(
  clientes$satisfaccion,
  levels = 1:5,
  ordered = TRUE
)
cat("Satisfaccion minima registrada:", as.character(min(clientes$satisfaccion)), "\n")

# Prueba de error de escritura en los niveles ("Postgrado" con t)
# Los valores que no coinciden con levels pasan a ser NA silenciosamente
educacion_typo <- factor(
  educacion_original,
  levels = c("Primaria", "Secundaria", "Universidad", "Postgrado"),
  ordered = TRUE
)
cat("Valores perdidos (NA) al escribir 'Postgrado' con t:", sum(is.na(educacion_typo)), "\n")

# 5. Resumen final de la clasificacion de variables
clasificacion <- c(
  cliente_id         = "Categorica nominal (identificador)",
  edad               = "Numerica continua (registrada en anos cumplidos, discreta en el archivo)",
  numero_hijos       = "Numerica discreta",
  genero             = "Categorica nominal (3 categorias)",
  nivel_educacion    = "Categorica ordinal",
  ciudad             = "Categorica nominal",
  ingresos_mensuales = "Numerica continua",
  satisfaccion       = "Categorica ordinal"
)

cat("\nResumen de la clasificacion:\n")
for (variable in names(clasificacion)) {
  cat(sprintf("  %-18s %s\n", variable, clasificacion[[variable]]))
}

| Diseño de muestreo | Tamaño ($n$) | Media estimada (Python) | Composición por ciudad | Diferencia frente al parámetro (`1330.72 USD`) |
|---|---|---|---|---|
| **Conveniencia (`head`)** | 20 | `1348.50 USD` | Quito: 20, Guayaquil: 0, Cuenca: 0 | `+17.78 USD` |
| **Aleatorio simple (`sample`, semilla 1)** | 20 | `1336.00 USD` | Cuenca: 8, Quito: 8, Guayaquil: 4 | `+5.28 USD` |
| **Estratificado (`groupby` + 5 %, semilla 1)** | 20 | `1287.50 USD` | Quito: 10, Guayaquil: 6, Cuenca: 4 | `-43.22 USD` |

*Variabilidad en 5 sorteos del muestreo aleatorio simple (semillas 1 a 5):* `1336.00 USD`, `1405.00 USD`, `1235.00 USD`, `1491.50 USD` y `1288.00 USD`.

## Preguntas Planteadas 


# 4. Conocimientos Adquiridos en la Práctica 
* Diferenciación práctica entre el tipo de almacenamiento inferido por el lenguaje (`dtype` / `str`) y la clasificación estadística real de una variable según su continuidad y orden.
* Configuración explícita de niveles jerárquicos en variables ordinales mediante `pd.CategoricalDtype` en Python y `factor(..., ordered = TRUE)` en R, previniendo errores silenciosos de valores perdidos (`NaN` / `NA`).
* Implementación comparativa de muestreo por conveniencia, muestreo aleatorio simple con control de semillas (`random_state` / `set.seed`) y muestreo estratificado proporcional por subgrupos.

# 5. Declaración de Uso de IA
Yo, Aidan Carrasco, declaro haber usado Inteligencia Artificial, especificamente Gemini para una redacción coherente del informe. El codigo fue hecho por mi persona con ayuda de páginas externas y adaptado a los ejercicios correspondientes.

# 6. Conclusiones 

1. **El criterio estadístico prevalece sobre el tipo inferido en código:** variables como `satisfaccion` ingresan como enteros (`int64` / `int`), pero representan escalas ordinales donde las distancias entre niveles no son equivalentes; a su vez, declarar explícitamente la jerarquía en variables como `nivel_educacion` evita ordenamientos alfabéticos erróneos y permite detectar errores tipográficos que generan valores faltantes silenciosos.

2. **El diseño del muestreo define la representatividad por encima del tamaño muestral:** el corte por conveniencia (`head(20)`) seleccionó exclusivamente clientes de Quito (`20/20`) debido al orden del archivo, demostrando que un sesgo sistemático de selección no se corrige agregando más filas bajo el mismo método.

3. **El muestreo estratificado y el control de la variabilidad respaldan mejores decisiones para la gerencia:** mientras el sorteo aleatorio simple mostró fluctuaciones importantes entre semillas (desde `1235.00 USD` hasta `1491.50 USD` frente al parámetro real de `1330.72 USD`), el diseño estratificado garantizó la presencia proporcional exacta de Quito (`10`), Guayaquil (`6`) y Cuenca (`4`), siendo la opción recomendada para los estudios comerciales de RetailAndina S.A.

# 7. Bibliografía

* Pandas Development Team. (2026). *pandas documentation: Categorical data & Indexing and selecting data (sample)*. https://pandas.pydata.org/docs/
* R Core Team. (2026). *R: A Language and Environment for Statistical Computing*. R Foundation for Statistical Computing. https://www.R-project.org/
* Wickham, H., François, R., Henry, L., Müller, K., & Vaughan, D. (2026). *dplyr: A Grammar of Data Manipulation (slice_sample)*. https://dplyr.tidyverse.org/